In [ ]:

# ================================================================
# GENERATIVE AI LABORATORY - EXPERIMENT 7
# TRANSFORMER-BASED LANGUAGE MODELS: GPT AND BERT
# ================================================================

# STEP 1: Install required libraries
%pip install -q transformers torch pandas

# STEP 2: Import required libraries
import torch
import pandas as pd
from transformers import pipeline, AutoTokenizer, AutoModelForMaskedLM

# STEP 3: Select the available device
device = 0 if torch.cuda.is_available() else -1

print("=" * 75)
print("GENERATIVE AI LABORATORY - EXPERIMENT 7")
print("TRANSFORMER-BASED LANGUAGE MODELS: GPT AND BERT")
print("=" * 75)

print("Device:", "GPU" if device == 0 else "CPU")

# ================================================================
# PART A: GPT-2 TEXT GENERATION
# ================================================================

print("\n" + "=" * 75)
print("PART A: GPT-2 TEXT GENERATION")
print("=" * 75)

# STEP 4: Load the GPT-2 text-generation pipeline
print("\nLoading GPT-2 model...")

generator = pipeline(
    "text-generation",
    model="gpt2",
    device=device
)

print("GPT-2 loaded successfully!")

# STEP 5: Define three different prompts
prompts = [
    "Generative AI is transforming education because",
    "In the future, artificial intelligence will",
    "The importance of learning programming is"
]

# STEP 6: Generate text for all three prompts
gpt_results = []

for i, prompt in enumerate(prompts, start=1):
    print(f"\n--- GPT Prompt {i} ---")
    print("Input:", prompt)

    output = generator(
        prompt,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        num_return_sequences=1,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    generated_text = output[0]["generated_text"]
    gpt_results.append({
        "Prompt": prompt,
        "Generated Text": generated_text
    })

    print("Generated Text:")
    print(generated_text)

# STEP 7: Compare generation settings
print("\n" + "=" * 75)
print("GPT-2: EFFECT OF GENERATION PARAMETERS")
print("=" * 75)

comparison_prompt = "Artificial intelligence can help society by"

# Greedy decoding: chooses the most likely next token
greedy_output = generator(
    comparison_prompt,
    max_new_tokens=40,
    do_sample=False,
    num_return_sequences=1,
    pad_token_id=generator.tokenizer.eos_token_id
)

print("\n1. Greedy Decoding")
print(greedy_output[0]["generated_text"])

# Sampling: allows alternative tokens to be selected
sampled_output = generator(
    comparison_prompt,
    max_new_tokens=40,
    do_sample=True,
    temperature=1.0,
    top_k=50,
    top_p=0.95,
    num_return_sequences=1,
    pad_token_id=generator.tokenizer.eos_token_id
)

print("\n2. Sampling (Temperature = 1.0, Top-k = 50, Top-p = 0.95)")
print(sampled_output[0]["generated_text"])

print("\nObservation:")
print("- Greedy decoding selects the highest-probability next token.")
print("- Sampling allows alternative tokens to be selected.")
print("- Results can vary between sampling runs.")
print("- Temperature changes the randomness of token selection.")

# ================================================================
# PART B: BERT MASKED-TOKEN PREDICTION
# ================================================================

print("\n" + "=" * 75)
print("PART B: BERT MASKED-TOKEN PREDICTION")
print("=" * 75)

# STEP 8: Load the pre-trained BERT model and tokenizer
print("\nLoading BERT model...")

bert_name = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)

bert_model = AutoModelForMaskedLM.from_pretrained(bert_name)

bert_model.eval()

fill_mask = pipeline(
    "fill-mask",
    model=bert_model,
    tokenizer=bert_tokenizer,
    device=device
)

print("BERT loaded successfully!")

# STEP 9: Provide two sentences containing [MASK]
masked_sentences = [
    "Artificial intelligence is changing the way people [MASK] information.",
    "The capital of France is [MASK]."
]

bert_mask_results = []

for i, sentence in enumerate(masked_sentences, start=1):
    print(f"\n--- BERT Masked Prediction {i} ---")
    print("Input:", sentence)

    predictions = fill_mask(sentence, top_k=5)

    print("Top 5 predictions:")

    for rank, item in enumerate(predictions, start=1):
        print(
            f"{rank}. {item['token_str'].strip()} "
            f"| Score: {item['score']:.4f}"
        )

    bert_mask_results.append({
        "Input Sentence": sentence,
        "Top Prediction": predictions[0]["token_str"].strip(),
        "Top Prediction Score": round(predictions[0]["score"], 4)
    })

# ================================================================
# PART C: BERT-FAMILY SENTIMENT CLASSIFICATION
# ================================================================

print("\n" + "=" * 75)
print("PART C: SENTIMENT CLASSIFICATION")
print("=" * 75)

# STEP 10: Load a pre-trained DistilBERT sentiment classifier.
# DistilBERT is a compact model derived from BERT.
print("\nLoading sentiment classification model...")

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

print("Sentiment model loaded successfully!")

sentiment_texts = [
    "I really enjoyed learning about transformer models.",
    "The application was confusing and did not work well.",
    "The lecture was informative and interesting."
]

sentiment_results = []

for text in sentiment_texts:
    result = sentiment_analyzer(text)[0]

    sentiment_results.append({
        "Input Text": text,
        "Predicted Sentiment": result["label"],
        "Confidence": round(result["score"], 4)
    })

    print("\nInput:", text)
    print("Sentiment:", result["label"])
    print("Confidence:", round(result["score"], 4))

# ================================================================
# PART D: GPT VS BERT COMPARISON
# ================================================================

print("\n" + "=" * 75)
print("PART D: GPT VS BERT COMPARISON")
print("=" * 75)

comparison_data = [
    ["Architecture", "Decoder-only", "Encoder-only"],
    ["Primary Purpose", "Text generation", "Language understanding"],
    ["Training Objective", "Next-token prediction", "Masked-token prediction"],
    ["Context", "Uses preceding context for generation",
     "Uses left and right context"],
    ["Main Demonstration", "Text completion", "Masked-word prediction"],
    ["Additional Task", "Creative text generation",
     "Classification using a BERT-family model"],
    ["Output", "Generated token sequence",
     "Predicted masked tokens or class labels"],
    ["Strength", "Generates coherent continuations",
     "Learns contextual word representations"],
    ["Limitation", "May generate inaccurate information",
     "Original BERT is not designed for free-form generation"]
]

comparison_df = pd.DataFrame(
    comparison_data,
    columns=["Parameter", "GPT-2", "BERT / BERT-family"]
)

print("\nComparison Table:")
print(comparison_df.to_string(index=False))

# ================================================================
# PART E: DISPLAY RESULTS IN TABLES
# ================================================================

print("\n" + "=" * 75)
print("PART E: RESULTS SUMMARY")
print("=" * 75)

print("\nGPT-2 Generation Results:")
gpt_df = pd.DataFrame(gpt_results)
print(gpt_df.to_string(index=False))

print("\nBERT Masked Prediction Results:")
mask_df = pd.DataFrame(bert_mask_results)
print(mask_df.to_string(index=False))

print("\nBERT-family Sentiment Classification Results:")
sentiment_df = pd.DataFrame(sentiment_results)
print(sentiment_df.to_string(index=False))

# STEP 11: Save outputs for the lab report
gpt_df.to_csv("experiment7_gpt_results.csv", index=False)
mask_df.to_csv("experiment7_bert_mask_results.csv", index=False)
sentiment_df.to_csv("experiment7_sentiment_results.csv", index=False)
comparison_df.to_csv("experiment7_comparison.csv", index=False)

print("\nSaved output files:")
print("1. experiment7_gpt_results.csv")
print("2. experiment7_bert_mask_results.csv")
print("3. experiment7_sentiment_results.csv")
print("4. experiment7_comparison.csv")

print("\n" + "=" * 75)
print("EXPERIMENT 7 COMPLETED SUCCESSFULLY")
print("=" * 75)


GENERATIVE AI LABORATORY - EXPERIMENT 7
TRANSFORMER-BASED LANGUAGE MODELS: GPT AND BERT
Device: CPU

PART A: GPT-2 TEXT GENERATION

Loading GPT-2 model...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

... (17 outputs left out)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentiment model loaded successfully!

Input: I really enjoyed learning about transformer models.
Sentiment: POSITIVE
Confidence: 0.9998

Input: The application was confusing and did not work well.
Sentiment: NEGATIVE
Confidence: 0.9998

Input: The lecture was informative and interesting.
Sentiment: POSITIVE
Confidence: 0.9998

PART D: GPT VS BERT COMPARISON

Comparison Table:
         Parameter                                 GPT-2                                     BERT / BERT-family
      Architecture                          Decoder-only                                           Encoder-only
   Primary Purpose                       Text generation                                 Language understanding
Training Objective                 Next-token prediction                                Masked-token prediction
           Context Uses preceding context for generation                            Uses left and right context
Main Demonstration                       Text completion     